In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans , DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC



In [ ]:
df = pd.read_csv("../data/raw/dataset.csv")
df_clean = df.copy()

In [ ]:
# Remove useless column
df_clean = df_clean.drop(columns=["CUST_ID"])

In [ ]:
df_clean['CREDIT_LIMIT'] = df_clean['CREDIT_LIMIT'].fillna(df_clean['CREDIT_LIMIT'].median())

In [ ]:
df_clean.to_csv("../data/processed/df_clean.csv", index=False)

In [ ]:
df_prepare_clustering = df_clean.copy()
x = df_prepare_clustering



In [ ]:
#LOGARITHMIC TRANSFORMATION

x_log = np.log1p(x)

x_log


In [ ]:
#skew after before

skew_comparison = pd.DataFrame({
    "before": x.skew(),
    "after_log1p": x_log.skew()
})


print(skew_comparison.round(3))

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(x_log)

X_scaled

In [ ]:
pca = PCA()

X_pca = pca.fit_transform(X_scaled)

print(X_pca[:2])

In [ ]:
print(pca.explained_variance_ratio_)

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

print(cumulative_variance)

# first 4 features represente the +80% of variance

In [ ]:
# PCA visualization
plt.figure(figsize=(8, 6))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    edgecolor='k'
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Transformed Data")

plt.grid()
plt.show()

In [ ]:
# I want to find the best combinaison of pca and k
results = []

for n_components in range(2, 6):  

    X = X_pca[:, :n_components]

    for k in range(2, 8):  

        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )

        labels = kmeans.fit_predict(X) #labels contains the cluster assigned to every client

        inertia = kmeans.inertia_
        silhouette = silhouette_score(X, labels)

        results.append({
            "n_components": n_components,
            "k": k,
            "inertia": inertia,
            "silhouette": silhouette
        })

results_df = pd.DataFrame(results)

print(results_df)

#the best combinaison is pca components = 2 and k = 3

In [ ]:


# Test k from 2 to 7
k_values = range(2, 8)

inertias = []
silhouette_scores = []

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_pca[:, :2])

    # Inertia
    inertias.append(kmeans.inertia_)

    # Silhouette
    silhouette_scores.append(
        silhouette_score(X_pca[:, :2], labels)
    )




In [ ]:

# ELBOW


plt.figure(figsize=(8, 6))

plt.plot(k_values, inertias, marker='o')

plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")


plt.grid()

plt.show()

In [ ]:
# SILHOUETTE

plt.figure(figsize=(8, 6))

plt.plot(k_values, silhouette_scores, marker='o')

plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")


plt.grid()

plt.show()

In [ ]:


#we use the best results (pca + k) we found to assign cluster to each client in dfclean
X_final = X_pca[:, :2]
best_k = 3

# Entraînement de K-means
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(X_final)

# Nombre de clients par cluster
cluster_counts = pd.Series(labels).value_counts().sort_index()

# Pourcentage de clients par cluster
cluster_percentages = (
    cluster_counts / len(labels) * 100
).round(2)

cluster_balance = pd.DataFrame({
    "count": cluster_counts,
    "percentage": cluster_percentages
})

print(cluster_balance)

In [ ]:
#visualisation de clusters

centroids = kmeans.cluster_centers_

plt.figure(figsize=(8, 6))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=labels,
    edgecolor='k'
)

plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    marker='X',
    s=250,
    edgecolor='black',
    linewidth=2,
    label='Centroids'
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title(f"K-means Clustering (k={k})")
plt.legend()
plt.show()

In [ ]:
df_clean["cluster_kmeans"] = labels

print(df_clean["cluster_kmeans"].value_counts().sort_index())

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples = 4

neighbors = NearestNeighbors(
    n_neighbors=min_samples
)

neighbors.fit(X_pca[:, :2])

distances, indices = neighbors.kneighbors(
    X_pca[:, :2]
)

k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_distances)

plt.xlabel("Points sorted by distance")
plt.ylabel(f"{min_samples}-NN distance")
plt.title("K-distance plot for choosing eps")

plt.show()

In [ ]:


X_dbscan = X_pca[:, :2]

results_dbscan = []

eps_values = [0.1, 0.2, 0.3, 0.5, 0.7]
min_samples_values = [4, 7 , 10, 11, 12 , 13,  15]

for eps in eps_values:

    for min_samples in min_samples_values:

        dbscan = DBSCAN(
            eps=eps,
            min_samples=min_samples
        )

        labels = dbscan.fit_predict(X_dbscan)

        # Nombre de clusters sans compter le bruit (-1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        # Nombre de points considérés comme bruit
        n_noise = list(labels).count(-1)

        # Calcul du silhouette uniquement s il y a au moins 2 clusters
        if n_clusters >= 2:
            mask = labels != -1

            silhouette = silhouette_score(
                X_dbscan[mask],
                labels[mask]
            )
        else:
            silhouette = None

        results_dbscan.append({
            "eps": eps,
            "min_samples": min_samples,
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "silhouette": silhouette
        })

results_dbscan_df = pd.DataFrame(results_dbscan)

print(results_dbscan_df)

In [ ]:
heatmap_data = results_dbscan_df.pivot(
    index="min_samples",
    columns="eps",
    values="silhouette"
)

plt.figure(figsize=(10, 6))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu"
)

plt.xlabel("eps")
plt.ylabel("min_samples")
plt.title("DBSCAN — Silhouette Score")
plt.show()

In [ ]:
#attach DBSCAN to df_clean

X_dbscan = X_pca[:, :2]

dbscan = DBSCAN(
    eps=0.3,
    min_samples=15
)

labels_dbscan = dbscan.fit_predict(X_dbscan)



In [ ]:
#Check the clusters and noise
cluster_counts = pd.Series(labels_dbscan).value_counts().sort_index()

print(cluster_counts)

In [ ]:
cluster_percentages = (
    cluster_counts / len(labels_dbscan) * 100
).round(2)

# noise_percentage = (
#     (labels_dbscan == -1).sum()
#     / len(labels_dbscan)
#     * 100
# )


dbscan_balance = pd.DataFrame({
    "count": cluster_counts,
    "percentage": cluster_percentages
})

print(dbscan_balance)

In [ ]:
#attach to dfclean
df_clean["cluster_dbscan"] = labels_dbscan

#anomaly indicator
df_clean["est_atypique_dbscan"] = (
    df_clean["cluster_dbscan"] == -1
)

df_clean[['cluster_kmeans' , 'cluster_dbscan', 'est_atypique_dbscan' ]]

In [ ]:
#Comparaison finale / calculer les 3 métriques pour K-means et DBSCAN

X_final = X_pca[:, :2]

# K-means
labels_kmeans = df_clean["cluster_kmeans"].values

silhouette_kmeans = silhouette_score(
    X_final,
    labels_kmeans
)

db_kmeans = davies_bouldin_score(
    X_final,
    labels_kmeans
)

ch_kmeans = calinski_harabasz_score(
    X_final,
    labels_kmeans
)

# DBSCAN
labels_dbscan = df_clean["cluster_dbscan"].values

# On retire le bruit pour les metriques de clustering
mask = labels_dbscan != -1

silhouette_dbscan = silhouette_score(
    X_final[mask],
    labels_dbscan[mask]
)

db_dbscan = davies_bouldin_score(
    X_final[mask],
    labels_dbscan[mask]
)

ch_dbscan = calinski_harabasz_score(
    X_final[mask],
    labels_dbscan[mask]
)

noise_percentage = (
    (labels_dbscan == -1).sum()
    / len(labels_dbscan)
    * 100
)

comparison = pd.DataFrame({
    "method": ["K-means", "DBSCAN"],
    "silhouette": [silhouette_kmeans, silhouette_dbscan],
    "davies_bouldin": [db_kmeans, db_dbscan],
    "calinski_harabasz": [ch_kmeans, ch_dbscan],
    "noise_percentage": [0, noise_percentage]
})

print(comparison)



In [ ]:
df_clean['cluster_final'] = df_clean['cluster_kmeans']

df_clean[['cluster_final']]

In [ ]:
#future story 4 
#calcule des moyens des futures pour chaque cluster in data reelle

df_cluser_means = df_clean.groupby('cluster_final').mean(numeric_only=True)
df_cluser_means

#cluster 0 -> HIGH_ACTIVITY
#cluster 1 -> CASH_ADVANCE
#cluster 3 -> LOWER_ACTIVITY

,BALANCE,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,CREDIT_LIMIT,PAYMENTS,cluster_kmeans,cluster_dbscan,est_atypique_dbscan
cluster_final,,,,,,,,,,
0,2266.791083,2203.375288,1414.467555,789.140411,1096.993912,6531.340899,2893.899180,0.0,-0.015667,0.015667
1,2174.726890,23.785284,19.025165,4.810189,2007.253219,3980.955231,1629.684882,1.0,0.814403,0.007407
2,334.366952,467.601664,150.124189,318.044198,54.874316,2695.116563,561.505351,2.0,-0.021996,0.030602


In [ ]:
clusters_names = {
    0 : 'HIGH_ACTIVITY',
    1 : 'CASH_ADVANCE',
    2 : 'LOWER_ACTIVITY'
    }

df_clean['target'] = df_clean['cluster_final'].map(clusters_names)

df_clean["target"].value_counts()

target
HIGH_ACTIVITY     3383
LOWER_ACTIVITY    3137
CASH_ADVANCE      2430
Name: count, dtype: int64

In [191]:
#saving
df_clean.to_csv("../data/processed/df_clean.csv", index=False)